## Persistent Landing - Delta Lake

The aim of this notebook is to convert all structure data we have into parquet files, in our case, it would be CSV files.

**Importing Useful Libraries**

In [2]:
import os
import boto3
import duckdb
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [3]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [48]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

In [59]:
# Create a specilized sub-bucket to store parquet files inside persistent-landing
s3.put_object(Bucket="landing-zone", Key="persistent-landing/csv-delta-lake/")

{'ResponseMetadata': {'RequestId': '189D72304BF876F9',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-checksum-crc32': 'AAAAAA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '189D72304BF876F9',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '9071',
   'x-ratelimit-remaining': '9071',
   'x-xss-protection': '1; mode=block',
   'date': 'Mon, 16 Mar 2026 22:16:20 GMT'},
  'RetryAttempts': 0},
 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
 'ChecksumCRC32': 'AAAAAA==',
 'ChecksumType': 'FULL_OBJECT'}

In [61]:
def ingest_with_duckdb(client, con, bucket, source_prefix="persistent-landing/csv/"):
    con.execute("INSTALL httpfs; LOAD httpfs;")

    paginator = client.get_paginator("list_objects_v2") # It returns objects in pages and not all at once.

    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            
            src_key = obj["Key"]

            if obj['Size'] == 0 and src_key.endswith("/"):
                continue

            # File name
            table_name = os.path.splitext(os.path.basename(src_key))[0]
            # S3 path
            s3_path = f"s3://{bucket}/{src_key}"
            output_path = f"s3://{bucket}/persistent-landing/csv-delta-lake/{table_name}.parquet"
            print(s3_path)
            print(output_path)

            print(f" DuckDB processing: {table_name}...")
            
            try:
                con.execute(f"""
                    COPY (
                        SELECT * FROM read_csv_auto('{s3_path}')
                    ) TO '{output_path}' (FORMAT PARQUET);
                """)
                print(f"✅ Saved to: {output_path}")
            except Exception as e:
                print(f"❌ DuckDB failed for {table_name}: {e}")

In [62]:
# Convert csv files into parquet files
ingest_with_duckdb(s3, con, "landing-zone", "persistent-landing/csv/")

s3://landing-zone/persistent-landing/csv/co2-emission-by-vehicles_1773688182749.csv
s3://landing-zone/persistent-landing/csv-delta-lake/co2-emission-by-vehicles_1773688182749.parquet
 DuckDB processing: co2-emission-by-vehicles_1773688182749...
✅ Saved to: s3://landing-zone/persistent-landing/csv-delta-lake/co2-emission-by-vehicles_1773688182749.parquet
s3://landing-zone/persistent-landing/csv/global_warming_dataset_1773688182804.csv
s3://landing-zone/persistent-landing/csv-delta-lake/global_warming_dataset_1773688182804.parquet
 DuckDB processing: global_warming_dataset_1773688182804...
✅ Saved to: s3://landing-zone/persistent-landing/csv-delta-lake/global_warming_dataset_1773688182804.parquet
s3://landing-zone/persistent-landing/csv/natural_disaster_tweets_1773688502706.csv
s3://landing-zone/persistent-landing/csv-delta-lake/natural_disaster_tweets_1773688502706.parquet
 DuckDB processing: natural_disaster_tweets_1773688502706...
✅ Saved to: s3://landing-zone/persistent-landing/csv-d

We can now try querying over these parquet filed.

In [63]:
print("🔎 Reading first 10 rows of co2-emission.parquet via DuckDB:")
table_path = "s3://landing-zone/persistent-landing/csv-delta-lake/co2-emission*.parquet"
df_view = con.execute(f"SELECT * FROM read_parquet('{table_path}') LIMIT 10").df()
df_view

🔎 Reading first 10 rows of co2-emission.parquet via DuckDB:


,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244
5,ACURA,RLX,MID-SIZE,3.5,6,AS6,Z,11.9,7.7,10.0,28,230
6,ACURA,TL,MID-SIZE,3.5,6,AS6,Z,11.8,8.1,10.1,28,232
7,ACURA,TL AWD,MID-SIZE,3.7,6,AS6,Z,12.8,9.0,11.1,25,255
8,ACURA,TL AWD,MID-SIZE,3.7,6,M6,Z,13.4,9.5,11.6,24,267
9,ACURA,TSX,COMPACT,2.4,4,AS5,Z,10.6,7.5,9.2,31,212


Next we read these parquet files and persist them as Delta Tables.

In [100]:
# Connection configuration for MinIO/S3
# Using the standard storage_options compatible with delta-rs
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

def write_parquets_as_delta_tables(client, con, bucket, storage_options, source_prefix="persistent-landing/csv-delta-lake/"):
    """
    Reads existing Parquet files and persists them as Delta Tables.
    This process creates the required folder structure and the _delta_log (Version 0).
    """
    paginator = client.get_paginator("list_objects_v2") # It returns objects in pages and not all at once.

    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            
            src_key = obj["Key"]

            if obj['Size'] == 0 and src_key.endswith("/"):
                continue

            # File name
            file_name = os.path.splitext(os.path.basename(src_key))[0]
            
            # Define source path (raw parquet) and target path (delta folder)
            source_path = f"s3://landing-zone/persistent-landing/csv-delta-lake/{file_name}.parquet"
            target_folder = f"s3://landing-zone/persistent-landing/csv-delta-lake/{file_name}/"
            
            print(f"📦 Finalizing Bronze Table: {file_name}...")
            
            try:
                # Load the parquet file into memory
                df = pl.read_parquet(source_path, storage_options=storage_options)
                
                # Write as Delta Table to create the versioned directory structure
                # Mode 'overwrite' ensures a fresh Version 0 is created
                write_deltalake(
                    target_folder,
                    df,
                    mode="overwrite",
                    storage_options=storage_options
                )
                print(f"✅ Successfully created Delta folder and log at: {target_folder}")
                s3.delete_object(
                    Bucket=bucket, 
                    Key=f"{source_prefix}{file_name}.parquet"
                )
            except Exception as e:
                print(f"❌ Failed to finalize {file_name}: {e}")

In [101]:
# Write parquet files as Delta Tables
write_parquets_as_delta_tables(s3, con, "landing-zone", storage_options, source_prefix="persistent-landing/csv-delta-lake/")

📦 Finalizing Bronze Table: co2-emission-by-vehicles_1773688182749...
✅ Successfully created Delta folder and log at: s3://landing-zone/persistent-landing/csv-delta-lake/co2-emission-by-vehicles_1773688182749/
📦 Finalizing Bronze Table: global_warming_dataset_1773688182804...
✅ Successfully created Delta folder and log at: s3://landing-zone/persistent-landing/csv-delta-lake/global_warming_dataset_1773688182804/
📦 Finalizing Bronze Table: natural_disaster_tweets_1773688502706...
✅ Successfully created Delta folder and log at: s3://landing-zone/persistent-landing/csv-delta-lake/natural_disaster_tweets_1773688502706/
📦 Finalizing Bronze Table: temperature_change_1773688502952...
✅ Successfully created Delta folder and log at: s3://landing-zone/persistent-landing/csv-delta-lake/temperature_change_1773688502952/
